# 02 · Member 2 analysis
**Owner:** _(name)_ · **Needs:** run `04_integration_validation.ipynb` first.

For each rule: write the hypothesis → test it on the full data → chart it with the rule as its title → save it to `outputs/` → add a row to `outputs/tables/rules/rules_02.csv` → run `combine_rules()`.

In [1]:
from pathlib import Path
import os, warnings; warnings.filterwarnings('ignore')
import duckdb, numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 60)

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW, PROC = ROOT / 'data' / 'raw', ROOT / 'data' / 'processed'
FIG, TAB = ROOT / 'outputs' / 'figures', ROOT / 'outputs' / 'tables'
RULES = TAB / 'rules'                       # one rules file per notebook -> combined into outputs/tables/rules_log.csv
for d in (RAW, PROC, FIG, TAB, RULES): d.mkdir(parents=True, exist_ok=True)
FULL    = PROC / 'payments_clean.parquet'   # all 10M rows (nothing removed) + derived columns, built by notebook 04
SAMPLE  = PROC / 'sample_20pct.parquet'     # 20% Bernoulli sample (seed 42), no free-text columns, still includes anomalies
ANOM    = PROC / 'anomalies.parquet'        # every labelled anomaly row, all columns
SCHEMA_VERSION = '2'                        # bump when derived columns change -> notebook 04 rebuilds the files

def connect(db=':memory:'):
    con = duckdb.connect(str(db))
    con.execute("SET TimeZone='UTC'")       # never skip: otherwise DuckDB shows laptop time (+08)
    con.execute("SET preserve_insertion_order=false")
    try: con.execute("SET enable_progress_bar=false")   # no progress widgets saved in the notebook ("Could not render ... widget" in viewers)
    except Exception: pass                           # (older setups without ipywidgets show no widget anyway)
    con.execute(f"SET temp_directory='{(PROC / 'duck_tmp').as_posix()}'")   # big joins spill to disk instead of crashing
    if os.environ.get('SWIFT_MEMORY'):  con.execute(f"SET memory_limit='{os.environ['SWIFT_MEMORY']}'")   # e.g. 4GB on an 8 GB laptop
    if os.environ.get('SWIFT_THREADS'): con.execute(f"SET threads={int(os.environ['SWIFT_THREADS'])}")
    return con

In [2]:
con = connect()

In [3]:
def sql(q):
    """Run SQL on the full data and return a pandas DataFrame."""
    return con.execute(q).df()
q = sql
def save_table(df, name):
    """Save a result table to outputs/tables/<name>.csv and return it for display."""
    df.to_csv(TAB / f'{name}.csv', index=False); return df

RULE_COLS = ['id', 'group', 'rule', 'test', 'value', 'verdict', 'evidence_files', 'interpretation']
VERDICTS = 'Confirmed (holds in >=99% of rows / effect matches) · Strong (95-99%) · Partial (70-95% or only in a subgroup) · Rejected'
def verdict_from_share(x):
    return 'Confirmed' if x >= 0.99 else 'Strong' if x >= 0.95 else 'Partial' if x >= 0.70 else 'Rejected'

def combine_rules():
    """Merge every outputs/tables/rules/rules_*.csv into one outputs/tables/rules_log.csv (shared schema)."""
    frames = []
    for f in sorted(RULES.glob('rules_*.csv')):
        d = pd.read_csv(f)
        ren = {'observed_evidence_table_or_chart': 'evidence_files', 'interpretation_real_world': 'interpretation', 'confidence': 'verdict'}
        d = d.rename(columns={k: v for k, v in ren.items() if k in d.columns and v not in d.columns})
        if 'rule' not in d.columns and {'when_condition', 'then_effect'} <= set(d.columns):
            d['rule'] = d.when_condition.astype(str) + ' -> ' + d.then_effect.astype(str)
        if 'value' not in d.columns and 'size' in d.columns: d['value'] = d['size']
        d['source'] = f.name; frames.append(d)
    if not frames: return pd.DataFrame(columns=RULE_COLS)
    out = pd.concat(frames, ignore_index=True)
    out = out[[c for c in RULE_COLS + ['source'] if c in out.columns] + [c for c in out.columns if c not in RULE_COLS + ['source']]]
    out.to_csv(TAB / 'rules_log.csv', index=False); return out

In [4]:
# Main view `p` = full data with the shared derived columns (see notebook 04, section 1)
assert FULL.exists() and (PROC / '_schema_version.txt').exists() and (PROC / '_schema_version.txt').read_text().strip() == SCHEMA_VERSION, \
    'Shared data missing or outdated: run notebooks/04_integration_validation.ipynb first'
con.execute(f"CREATE OR REPLACE VIEW p AS SELECT * FROM read_parquet('{FULL.as_posix()}')")
# Local currency of each BIC country = the currency its senders use most (inferred from this dataset, no external lookup)
con.execute("""CREATE OR REPLACE TABLE local_ccy AS
  SELECT o_cc AS cc, arg_max(CURRENCY_CODE, n) AS local_ccy FROM
  (SELECT o_cc, CURRENCY_CODE, count(*) n FROM p GROUP BY ALL) GROUP BY o_cc""")
print(f"{con.execute('SELECT count(*) FROM p').fetchone()[0]:,} payments loaded")

10,000,000 payments loaded


## Feature group: _(e.g. Routing)_
**Hypothesis:** _…_

In [5]:
# Example query (full data, fast with DuckDB):
sql("SELECT TRANSACTION_TYPE_CODE, count(*) n, median(TOTAL_DURATION) med_duration FROM p GROUP BY 1")

,TRANSACTION_TYPE_CODE,n,med_duration
0,Indirect,3260758,281.0
1,Covered,415962,383.0
2,Direct,6323280,39.0


In [6]:
# Pandas sample for charts / models (includes anomalies: filter anomaly == 'NONE' for normal-only work)
s = pd.read_parquet(SAMPLE); s.head()

,EPOCH_SECONDS,CURRENCY_CODE,USD_AMOUNT,ORDERING_BIC8,BENEFICIARY_BIC8,BIC8_PATH,TOTAL_DURATION,CONTINUITY_QFLAG,CURRENCY_CONVERSION_COUNT,PARTICIPANT_COUNT,TRANSACTION_TYPE_CODE,ORDERING_ACCOUNT,BENEFICIARY_ACCOUNT,ORDERING_TYPE,BENEFICIARY_TYPE,ANOMALY_TYPE,rid,ts,o_cc,b_cc,pair,anomaly,cross_border,path_len,path_gap,o_det_cc,b_det_cc,o_det_acct,b_det_acct
0,1686668639,EUR,1447.90,0412FRP0,0031TWT0,0412FRP0;0023FIH0;0035LUL0;0071KRS0;0005BHB0;0...,1900,Y,3,6,Covered,_FAKE_FR45041200000000349730,_FAKE_89V7929,B,B,None,7,2023-06-13 15:03:59+00:00,FR,TW,B2B,NONE,True,6,False,FR,,FR45041200000000349730,89V7929
1,1688034786,USD,1323.45,0100IT20,0005BDD0,1107US30;0005BDD0,12,Y,1,2,Direct,_FAKE_IT28010000000000619567,_FAKE_06-20BDT385,C,C,None,10,2023-06-29 10:33:06+00:00,IT,BD,C2C,NONE,True,2,False,,,IT28010000000000619567,06-20BDT385
2,1684938706,DKK,123814.40,0003DKK0,0004IMD0,0003DKK0;0004IMD0,26,Y,1,2,Direct,_FAKE_DK88000300000000297296,_FAKE_41GBP/9 1 62,C,B,None,14,2023-05-24 14:31:46+00:00,DK,IM,C2B,NONE,True,2,False,DK,IN,DK88000300000000297296,41GBP/9 1 62
3,1687888237,DZD,8.39,0012DZA0,0002LUL0,0012DZA0;0007IMD0;0215GB20;...;0002LUL0,39,N,2,4,Indirect,_FAKE_300D703,_FAKE_LU62000200000000889164,B,C,None,20,2023-06-27 17:50:37+00:00,DZ,LU,B2C,NONE,True,5,True,DZ,CN,300D703,LU62000200000000889164
4,1690185456,GBP,251.15,0004IMD0,0086KRS0,0004IMD0;0077CHZ0;0086KRS0,322,Y,2,3,Indirect,_FAKE_41GBP/9 1 62,_FAKE_28 /46/89-/.,B,C,None,22,2023-07-24 07:57:36+00:00,IM,KR,B2C,NONE,True,3,False,IM,,41GBP/9 1 62,28 /46/89-/.


**Findings:** _rule ID, statement, evidence, economic meaning_

In [7]:
# Rules for this notebook (same columns as the other notebooks), then rebuild the combined log
# pd.DataFrame([dict(id='R-RT1', group='Routing', rule='...', test='...', value='...', verdict='Confirmed', evidence_files='...', interpretation='...')],
#              columns=RULE_COLS).to_csv(RULES / 'rules_02.csv', index=False)
combine_rules()

,id,group,rule,test,value,verdict,evidence_files,interpretation,source,when_condition,then_effect,size,scope_rows_pct,exceptions,rejected_alternative
0,A2-example,Anomalies,Payment labelled DUPLICATE_PAYMENT and sender ...,NaN,median gap = ?? s,Confirmed / Strong / Partial / Rejected,a_duplicates_vs_control.csv; A2_duplicate_gap.png,INTERPRETATION: back-office race condition re-...,rules_03_anomalies.csv,Payment labelled DUPLICATE_PAYMENT and sender ...,"An earlier payment exists with same sender, be...",median gap = ?? s,?? of ?? duplicates (??%),C senders: different sender account (partner) ...,Exact copy of the row (rejected: timestamps di...
1,R-CCY1,Currency,Payments are made in the sender country's loca...,full data; local currency inferred in-sample,89.4% sender-local; only 4.2% use the receiver...,Partial (exceptions not yet explained),ccy1_currency_choice.png,The sender's bank debits in its home currency;...,rules_04_core.csv,NaN,NaN,NaN,NaN,NaN,NaN
2,R-CCY2,Currency,Direct payments: 1 conversion if sender and re...,prediction accuracy on all Direct rows,97.8%,Strong,ccy2_direct_conversion_rule.png,"With only two banks, FX happens once, and only...",rules_04_core.csv,NaN,NaN,NaN,NaN,NaN,NaN
3,R-CCY3,Currency,Conversions never exceed the number of hops (p...,share of all rows,100.00%,Confirmed,ccy3_conversions_by_hops.csv,Each correspondent bank can convert at most once,rules_04_core.csv,NaN,NaN,NaN,NaN,NaN,NaN
4,R-AMT1,Amount,Amounts are log-normal: ln(amount) is normal w...,skew / excess kurtosis / sd of ln residual,"skew -0.016, kurtosis 0.024, sd 2.08",Confirmed,amt1_lognormal_by_pair.png,Payment sizes built from many multiplying fact...,rules_04_core.csv,NaN,NaN,NaN,NaN,NaN,NaN
5,R-AMT2,Amount,Each business party (sender or receiver) multi...,"ln effect within sender country, 95% cluster-b...","sender 1.999 [1.980, 2.019]; receiver 2.002 [1...",Confirmed,amt2_business_multiplier.png,Businesses pay invoices and payroll; consumers...,rules_04_core.csv,NaN,NaN,NaN,NaN,NaN,NaN
6,R-AMT3,Amount,The sender's country sets the base amount; the...,R² sender country vs extra R² of receiver country,0.131 vs 0.0003,Confirmed,amt3_country_base_amount.png,Payment size scales with the income level of t...,rules_04_core.csv,NaN,NaN,NaN,NaN,NaN,NaN
7,R-BC1,Business/consumer,Sender type and receiver type are assigned ind...,"chi-square p, Cramér's V","p = 0.67, V = 0.0003",Confirmed,bc1_type_independence.png,Who pays and who gets paid are separate draws,rules_04_core.csv,NaN,NaN,NaN,NaN,NaN,NaN
8,R-BC2,Business/consumer,Timestamps follow the sender's local working d...,"share 07:00-19:59 local vs UTC; V(hour, sender...",74.5% local vs 65.0% UTC; V = 0.024,Strong (time zone is approximate),bc2_local_hour_profile.png,Payments are entered and batched in office hours,rules_04_core.csv,NaN,NaN,NaN,NaN,NaN,NaN
9,R-BC3,Business/consumer,Business sender accounts send more payments th...,"payments per (sender BIC, account) over the da...",B 44.8 vs C 9.9 (x4.5),Confirmed,bc3_activity_per_account.png,Companies pay many suppliers and staff,rules_04_core.csv,NaN,NaN,NaN,NaN,NaN,NaN
